# Parte 5 — Questões de negócio: Nível 2 (fatores de risco)

**Q5 a Q7.** Mesmas regras do Nível 1: taxa de letalidade = % de acidentes fatais do
grupo, sempre comparada à taxa global de **7,16%**, com volume mínimo declarado e
interpretação sem linguagem causal.


## 0. Base analítica da Parte 4

Lida de `dados_tratados/base_analitica_prf_2023_2025.csv` — a base já vem no
repositório. Se o arquivo não existir, rode antes o notebook da Parte 4.


In [1]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd


def localizar_raiz() -> Path:
    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / "dados_brutos").is_dir():
            return candidato
    raise FileNotFoundError("Abra o notebook da raiz do projeto ou de 'notebooks/'.")


RAIZ = localizar_raiz()
DIR_BRUTOS = RAIZ / "dados_brutos"
DIR_TRATADOS = RAIZ / "dados_tratados"
DIR_RESULTADOS = RAIZ / "resultados"
for pasta in (DIR_TRATADOS, DIR_RESULTADOS):
    pasta.mkdir(parents=True, exist_ok=True)

SEPARADOR = ";"
ENCODING_SAIDA = "utf-8-sig"
ANOS = (2023, 2024, 2025)
print(RAIZ)


/home/user/PRF-pandas-e-Plotly---2023-2025


In [2]:
ARQ_ANALITICA = DIR_TRATADOS / "base_analitica_prf_2023_2025.csv"
df = pd.read_csv(ARQ_ANALITICA, sep=SEPARADOR, encoding=ENCODING_SAIDA,
                 dtype={"id": "string"}, low_memory=False)
df["data_inversa"] = pd.to_datetime(df["data_inversa"], errors="coerce")

TAXA_GLOBAL = df["acidente_fatal"].mean()
print(f"Base: {df.shape[0]:,} acidentes | taxa global de fatais: {TAXA_GLOBAL:.4%}")


Base: 213,497 acidentes | taxa global de fatais: 7.1631%


In [3]:
# --- funções de apoio usadas nas questões ---
def indicadores(base: pd.DataFrame, coluna: str, minimo: int = 0) -> pd.DataFrame:
    """Acidentes, fatais, mortos e taxa de letalidade por categoria.

    Neste exercício, taxa de letalidade = % de acidentes fatais do grupo
    (definição do enunciado), comparada com a taxa global.
    """
    tab = base.groupby(coluna).agg(
        acidentes=("acidente_fatal", "size"),
        fatais=("acidente_fatal", "sum"),
        mortos=("mortos", "sum"),
    )
    tab = tab[tab["acidentes"] >= minimo]
    tab["taxa_letalidade"] = tab["fatais"] / tab["acidentes"]
    tab["lift"] = tab["taxa_letalidade"] / TAXA_GLOBAL
    return tab.sort_values("taxa_letalidade", ascending=False)


def fmt(tab: pd.DataFrame) -> pd.DataFrame:
    """Formata percentuais para leitura (ignora colunas ausentes)."""
    saida = tab.copy()
    if "taxa_letalidade" in saida.columns:
        saida["taxa_letalidade"] = saida["taxa_letalidade"].map("{:.2%}".format)
    if "lift" in saida.columns:
        saida["lift"] = saida["lift"].map("{:.2f}".format)
    return saida


---
## Q5 — Lift por `tipo_acidente` (taxa do tipo ÷ taxa global),
só tipos com pelo menos 100 registros. Qual eleva mais a chance de morte?


In [4]:
q5 = indicadores(df, "tipo_acidente", minimo=100)
display(fmt(q5)[["acidentes", "fatais", "mortos", "taxa_letalidade", "lift"]])

excluidos = sorted(set(df["tipo_acidente"].dropna()) - set(q5.index))
print("Volume mínimo aplicado: 100 acidentes")
print(f"Tipos excluídos pelo filtro: {excluidos or 'nenhum'}")
print(f"Tipos analisados: {len(q5)}")
print()
print("Destaques:")
for posicao in list(range(3)) + [len(q5) - 1]:
    nome = q5.index[posicao]
    print(f"  {posicao + 1:>2}º de {len(q5)} — {nome[:34]:<34} "
          f"{q5['taxa_letalidade'].iloc[posicao]:>6.2%}  "
          f"lift {q5['lift'].iloc[posicao]:.2f}")


,acidentes,fatais,mortos,taxa_letalidade,lift
tipo_acidente,,,,,
Colisão frontal,14233,4243,5655,29.81%,4.16
Atropelamento de Pedestre,9361,2711,2757,28.96%,4.04
Colisão lateral sentido oposto,6200,541,690,8.73%,1.22
Eventos atípicos,871,71,73,8.15%,1.14
Atropelamento de Animal,3549,232,247,6.54%,0.91
Saída de leito carroçável,31248,1812,2078,5.80%,0.81
Colisão com objeto,15090,784,865,5.20%,0.73
Colisão transversal,27165,1232,1398,4.54%,0.63
Tombamento,18498,776,840,4.20%,0.59


Volume mínimo aplicado: 100 acidentes
Tipos excluídos pelo filtro: ['Sinistro pessoal de trânsito']
Tipos analisados: 16

Destaques:
   1º de 16 — Colisão frontal                    29.81%  lift 4.16
   2º de 16 — Atropelamento de Pedestre          28.96%  lift 4.04
   3º de 16 — Colisão lateral sentido oposto      8.73%  lift 1.22
  16º de 16 — Incêndio                            0.08%  lift 0.01


**Interpretação.** `Colisão frontal` (**29,81%**, lift **4,16**) e `atropelamento de pedestre`
(**28,96%**, lift **4,04**) têm a taxa mais alta — cerca de **quatro vezes** a global (7,16%) —
e somam **6.954 dos 15.293** fatais (45%). Volume mínimo: **100 registros** (1 tipo fora).


**Detalhamento.** A queda para o terceiro colocado é abrupta: `colisão lateral sentido
oposto` fica em 8,73% (lift 1,22). A letalidade **não acompanha a frequência**: `colisão
traseira` é o tipo mais comum (41.237 casos) e tem lift 0,58.


---
## Q6 — Top 5 causas por lift, declarando o volume mínimo.
Álcool ou ultrapassagem indevida entram?


In [5]:
q6 = indicadores(df, "causa_acidente", minimo=100)
display(fmt(q6.head(10))[["acidentes", "fatais", "taxa_letalidade", "lift"]])

print("Volume mínimo aplicado: 100 acidentes")
print(f"Causas analisadas: {len(q6)} de {df['causa_acidente'].nunique()} do vocabulário")
print(f"Causas excluídas pelo filtro: {df['causa_acidente'].nunique() - len(q6)}")
print()
top5 = q6.head(5).index.tolist()
print("TOP 5 POR LIFT:")
for i, nome in enumerate(top5, 1):
    print(f"  {i}. {nome[:52]:<52} {q6.loc[nome, 'taxa_letalidade']:>6.2%} "
          f"(lift {q6.loc[nome, 'lift']:.2f})")


,acidentes,fatais,taxa_letalidade,lift
causa_acidente,,,,
Suicídio (presumido),438,223,50.91%,7.11
Pedestre andava na pista,1899,787,41.44%,5.79
Entrada inopinada do pedestre,2034,600,29.50%,4.12
Transitar na contramão,7301,2114,28.95%,4.04
Pedestre cruzava a pista fora da faixa,1692,436,25.77%,3.60
Área urbana sem a presença de local apropriado para a travessia de pedestres,233,47,20.17%,2.82
Ultrapassagem Indevida,5093,845,16.59%,2.32
Pedestre - Ingestão de álcool/ substâncias psicoativas,323,52,16.10%,2.25
Iluminação deficiente,485,78,16.08%,2.25


Volume mínimo aplicado: 100 acidentes
Causas analisadas: 57 de 75 do vocabulário
Causas excluídas pelo filtro: 18

TOP 5 POR LIFT:
  1. Suicídio (presumido)                                 50.91% (lift 7.11)
  2. Pedestre andava na pista                             41.44% (lift 5.79)
  3. Entrada inopinada do pedestre                        29.50% (lift 4.12)
  4. Transitar na contramão                               28.95% (lift 4.04)
  5. Pedestre cruzava a pista fora da faixa               25.77% (lift 3.60)


In [6]:
# posição das causas citadas na pergunta — nos DOIS rankings, para não haver
# ambiguidade: o filtrado (57 causas, o que a análise compara) e o vocabulário
# inteiro (75 causas, que inclui as de volume baixo).
completo = indicadores(df, "causa_acidente", minimo=0)
citadas = q6[q6.index.str.contains("lcool|ltrapassagem", case=False, regex=True)]
print("CAUSAS CITADAS NA PERGUNTA — posição nos dois rankings:")
print(f"  {'causa':<50} {'filtrado':>9} {'completo':>9}  lift")
for nome, linha in citadas.iterrows():
    pos_filtrada = q6.index.get_loc(nome) + 1
    pos_completa = completo.index.get_loc(nome) + 1
    entra = "SIM" if nome in top5 else "NÃO"
    acima = "acima" if linha["lift"] > 1 else "ABAIXO"
    print(f"  {nome[:50]:<50} {pos_filtrada:>8}º {pos_completa:>8}º  "
          f"{linha['lift']:>5.2f} | top5: {entra} | {acima} da média")

assert "Ultrapassagem Indevida" in q6.index, "Ultrapassagem deveria passar no filtro"
print("\nUltrapassagem Indevida passa no filtro de volume e entra na análise.")


CAUSAS CITADAS NA PERGUNTA — posição nos dois rankings:
  causa                                               filtrado  completo  lift
  Ultrapassagem Indevida                                    7º        8º   2.32 | top5: NÃO | acima da média
  Pedestre - Ingestão de álcool/ substâncias psicoat        8º        9º   2.25 | top5: NÃO | acima da média
  Ingestão de álcool e/ou substâncias psicoativas pe       16º       22º   1.29 | top5: NÃO | acima da média
  Ingestão de álcool pelo condutor                         38º       49º   0.63 | top5: NÃO | ABAIXO da média

Ultrapassagem Indevida passa no filtro de volume e entra na análise.


**Interpretação.** O top 5 por lift é: **suicídio (presumido)** (50,91%, lift **7,11**),
**pedestre andava na pista** (41,44%), **entrada inopinada do pedestre** (29,50%), **transitar
na contramão** (28,95%) e **pedestre cruzava fora da faixa** (25,77%). Mínimo: **100** (57 de 75).


**Álcool e ultrapassagem indevida não entram no top 5** — mas ambas ficam **acima** da
média global, o que responde à pergunta de forma mais útil que um simples sim/não:

| Causa | Acidentes | Taxa | Lift | No top 5? |
|---|---|---|---|---|
| Ultrapassagem Indevida | 5.093 | 16,59% | 2,32 | não (7º) |
| Pedestre – ingestão de álcool | 323 | 16,10% | 2,25 | não (8º) |
| Ingestão de álcool pelo pedestre | 217 | 9,22% | 1,29 | não (16º) |
| Ingestão de álcool pelo condutor | 11.139 | 4,53% | **0,63** | não (38º) |


**Detalhamento.** O contraste **dentro** do tema álcool é o achado: as duas categorias de
álcool **do pedestre** ficam acima da média (lift 2,25 e 1,29), enquanto a de álcool **do
condutor** cai para 0,63 (**abaixo** da média). São verbetes distintos do vocabulário da base,
**associados a** letalidades diferentes — e o dado não isola o papel da substância em nenhum
deles.

`suicídio (presumido)` lidera com folga, mas é uma categoria de **438 registros** e de
natureza distinta das demais: não descreve uma falha de condução, e sim uma intenção.
Tratá-la como "fator de risco de trânsito" seria erro de interpretação — o correto é
reconhecê-la como um fenômeno à parte, que o ranking por lift expõe mas não explica.


---
## Q7 — `tracado_via`: reta ou curva tem taxa pior?
Só traçados com mais de 500 acidentes. Um registro pode ter mais de um traçado —
escolher e justificar como tratar.


In [7]:
# diagnóstico do problema multi-valor antes de decidir o tratamento
componentes = df["tracado_via"].str.count(";") + 1
print("Quantidade de traçados por registro:")
print(componentes.value_counts().sort_index().to_string())
print()
print(f"Registros com 1 traçado : {int((componentes == 1).sum()):,} "
      f"({(componentes == 1).mean():.1%})")
print(f"Registros com 2 ou mais : {int((componentes > 1).sum()):,} "
      f"({(componentes > 1).mean():.1%})")
print(f"Máximo de traçados num mesmo registro: {int(componentes.max())}")


Quantidade de traçados por registro:
tracado_via
1    159746
2     47939
3      5210
4       532
5        62
6         7
8         1

Registros com 1 traçado : 159,746 (74.8%)
Registros com 2 ou mais : 53,751 (25.2%)
Máximo de traçados num mesmo registro: 8


In [8]:
# tratamento escolhido: explode — cada traçado conta em todos os registros que o possuem.
# Justificativa no texto abaixo da célula.
explodido = (
    df[["tracado_via", "acidente_fatal", "mortos"]]
    .assign(tracado=df["tracado_via"].str.split(";"))
    .explode("tracado")
)
explodido["tracado"] = explodido["tracado"].str.strip()

print(f"Linhas após o explode: {len(explodido):,} (originais: {len(df):,})")


Linhas após o explode: 273,742 (originais: 213,497)


In [9]:
q7 = indicadores(explodido, "tracado", minimo=501)
display(fmt(q7)[["acidentes", "fatais", "mortos", "taxa_letalidade", "lift"]])

# o que o filtro de volume deixa de fora, declarado na hora
excluidos_q7 = (explodido.groupby("tracado").agg(acidentes=("acidente_fatal", "size"))
                .query("acidentes < 501").sort_values("acidentes", ascending=False))
print("Volume mínimo aplicado: mais de 500 acidentes (>= 501)")
print(f"Traçados exibidos: {q7.shape[0]} | descartados pelo filtro: {len(excluidos_q7)}")
for nome, linha in excluidos_q7.iterrows():
    sub = explodido[explodido["tracado"] == nome]
    print(f"  fora do corte: {nome} — {int(linha['acidentes'])} acidentes, "
          f"{int(sub['acidente_fatal'].sum())} fatais "
          f"(taxa {sub['acidente_fatal'].mean():.2%})")
print()
reta, curva = q7.loc["Reta"], q7.loc["Curva"]
print(f"Reta : {reta['taxa_letalidade']:.2%} (lift {reta['lift']:.2f}) — "
      f"{int(reta['acidentes']):,} acidentes")
print(f"Curva: {curva['taxa_letalidade']:.2%} (lift {curva['lift']:.2f}) — "
      f"{int(curva['acidentes']):,} acidentes")
print(f"Diferença: {curva['taxa_letalidade'] - reta['taxa_letalidade']:+.2%} "
      f"(curva {'pior' if curva['taxa_letalidade'] > reta['taxa_letalidade'] else 'melhor'})")

assert curva["taxa_letalidade"] > reta["taxa_letalidade"]


,acidentes,fatais,mortos,taxa_letalidade,lift
tracado,,,,,
Ponte,2401,258,320,10.75%,1.50
Declive,23349,2300,2851,9.85%,1.38
Aclive,17613,1524,1798,8.65%,1.21
Curva,39363,3223,3879,8.19%,1.14
Reta,154920,11383,13163,7.35%,1.03
Em Obras,5050,322,388,6.38%,0.89
Desvio Temporário,1002,55,67,5.49%,0.77
Viaduto,3647,142,150,3.89%,0.54
Interseção de Vias,15244,528,593,3.46%,0.48


Volume mínimo aplicado: mais de 500 acidentes (>= 501)
Traçados exibidos: 11 | descartados pelo filtro: 1
  fora do corte: Túnel — 125 acidentes, 5 fatais (taxa 4.00%)

Reta : 7.35% (lift 1.03) — 154,920 acidentes
Curva: 8.19% (lift 1.14) — 39,363 acidentes
Diferença: +0.84% (curva pior)


**Como foi tratado o multi-valor.** `tracado_via` é uma lista separada por `;`, e
**53.751 registros (25,2%) têm mais de um traçado**, chegando a 8 no mesmo acidente. As
três opções possíveis:

| Tratamento | Efeito |
|---|---|
| Manter só o 1º valor | simples, mas descarta 25% da informação e depende da ordem em que a PRF gravou |
| One-hot de flags | correto, porém gera 11 colunas em vez de 1 para responder uma pergunta de ranking |
| **Explode (escolhido)** | cada traçado conta em **todos** os registros que o possuem |

Escolhi o **explode** porque a pergunta é sobre gravidade *associada a* um traçado, e um
acidente em `Reta;Declive` genuinamente pertence às duas categorias. **Consequência
declarada:** os grupos **não formam partição** — as linhas somam 273.742 contra 213.497
acidentes, e as porcentagens **não somam 100%**. Cada taxa deve ser lida isoladamente.

**Interpretação.** **Curva tem taxa pior que reta**, mas a diferença é modesta: **8,19%
contra 7,35%** (+0,84 p.p., lift 1,14 contra 1,03). A resposta mais relevante, porém, é que
**nem curva nem reta são os piores traçados**: `Ponte` lidera com **10,75%** e `Declive` vem
em seguida com **9,85%**. Volume mínimo: **500 acidentes**.


**Detalhamento.** Os traçados de maior taxa são **associados** a rampas e a obras — `Aclive`
(8,65%), `Declive` (9,85%), `Ponte` (10,75%) e `Em Obras` (6,38%) — e os de menor taxa são
`Rotatória` (2,06%, lift 0,29) e `Retorno Regulamentado` (3,30%). A base não traz velocidade,
iluminação nem exposição: o dado descreve a associação sem explicá-la.


---
## Fechamento do Nível 2

| Questão | Achado |
|---|---|
| Q5 | Colisão frontal (lift 4,16) e atropelamento de pedestre (lift 4,04) elevam mais a chance de morte |
| Q6 | Top 5: suicídio (presumido), pedestre na pista, entrada inopinada, contramão, pedestre fora da faixa. Álcool no pedestre (2,25) e ultrapassagem (2,32) ficam acima da média, mas fora do top 5; álcool no condutor fica **abaixo** (0,63) |
| Q7 | Curva (8,19%) pior que reta (7,35%), mas ponte (10,75%) e declive (9,85%) são piores ainda; explode declarado como não-partição |
